In [1]:
import joblib
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score
from utils import custom_feature_engineering 
from sklearn.ensemble import VotingClassifier


pipeline_path = "/Users/hayashieijun/Desktop/airline-satisfaction-classification/pipelines/logistic_regression_pipeline.pkl"
loaded_pipeline = joblib.load(pipeline_path)


test_data = pd.read_csv('/Users/hayashieijun/Desktop/airline-satisfaction-classification/data/test.csv')
x_test = test_data.drop(columns=['Unnamed: 0', 'id'])
y_test = test_data['satisfaction'].map({'neutral or dissatisfied': 0, 'satisfied': 1})


preds_baseline = loaded_pipeline.predict(x_test)
print("--- Baseline Pipeline Performance ---")
print(f"Accuracy: {accuracy_score(y_test, preds_baseline):.4f}")
print(f"F1 Macro: {f1_score(y_test, preds_baseline, average='macro'):.4f}")

--- Baseline Pipeline Performance ---
Accuracy: 0.8693
F1 Macro: 0.8668


In [2]:
xgb_pipe = joblib.load("../pipelines/xgb_pipeline.pkl")
lr_pipe = joblib.load("../pipelines/logistic_regression_pipeline.pkl")
svm_pipe = lr_pipe = joblib.load("../pipelines/svm_pipeline.pkl")

/Users/hayashieijun/Desktop/anaconda3/lib/python3.13/pickle.py:1760: UserWarning: [09:02:53] WARNING: /Users/runner/work/xgboost/xgboost/src/gbm/../common/error_msg.h:83: If you are loading a serialized model (like pickle in Python, RDS in R) or
configuration generated by an older version of XGBoost, please export the model by calling
`Booster.save_model` from that version first, then load it back in current version. See:

    https://xgboost.readthedocs.io/en/stable/tutorials/saving_model.html

for more details about differences between saving model and serializing.

  setstate(state)


In [3]:
voting_clf = VotingClassifier(
    estimators=[
        ('lr', lr_pipe),  
        ('xgb', xgb_pipe)
    ],
    voting='soft'
)

In [ ]:
train_df = pd.read_csv("../data/train.csv", index_col=0)
X_train = train_df.drop(columns=["satisfaction", "id"])
y_train = train_df["satisfaction"]

voting_clf.fit(X_train, y_train)

In [21]:
test_df = pd.read_csv("../data/test.csv", index_col=0)
X_test = test_df.drop(columns=["satisfaction", "id"])
y_test = test_df["satisfaction"]

In [ ]:
preds = voting_clf.predict(X_test)

print("Soft voting F1 Macro:", f1_score(y_test, preds, average="macro"))
print("\nClassification Report:\n")
print(classification_report(y_test, preds))